In [4]:
# Import required libraries for database + data handling
import pandas as pd  # Data manipulation (tables like Excel)
import mysql.connector  # Connect Python to MySQL database

# Create connection to MySQL database
conn = mysql.connector.connect(
    host="localhost",        # Your MySQL server (local machine)
    user="root",             # Your MySQL username
    password="12three",# Replace with your MySQL password
    database="telecom_lab"   # Your telecom database
)

# SQL query to join KPI data with cell metadata
query = """
SELECT 
    k.cell_id,
    k.timestamp,
    k.rrc_connected_users,
    k.prb_utilization,
    k.throughput_mbps,
    k.latency_ms,
    k.packet_loss,
    d.region,
    d.technology,
    d.band
FROM cell_kpi k
JOIN dim_cell d
ON k.cell_id = d.cell_id;
"""

# Load data into Pandas DataFrame
df = pd.read_sql(query, conn)

# Show first 5 rows
df.head()

C:\Users\azeem\AppData\Local\Temp\ipykernel_7756\2576029008.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,cell_id,timestamp,rrc_connected_users,prb_utilization,throughput_mbps,latency_ms,packet_loss,region,technology,band
0,101,2026-06-13 10:00:00,120,75.5,85.2,18.3,0.2,Riyadh,LTE,1800
1,102,2026-06-13 10:00:00,95,60.1,92.5,15.1,0.1,Riyadh,LTE,2100
2,103,2026-06-13 10:00:00,180,88.7,70.4,25.6,0.5,Riyadh,NR,3500


In [3]:
!pip install mysql-connector-python

   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
    --------------------------------------- 0.3/18.2 MB ? eta -:--:--
   - -------------------------------------- 0.8/18.2 MB 2.8 MB/s eta 0:00:07
   -- ------------------------------------- 1.3/18.2 MB 2.7 MB/s eta 0:00:07
   ---- ----------------------------------- 2.1/18.2 MB 2.9 MB/s eta 0:00:06
   -------- ------------------------------- 3.7/18.2 MB 3.9 MB/s eta 0:00:04
   ---------- ----------------------------- 4.7/18.2 MB 4.1 MB/s eta 0:00:04
   ------------- -------------------------- 6.3/18.2 MB 4.6 MB/s eta 0:00:03
   --------------- ------------------------ 7.1/18.2 MB 4.6 MB/s eta 0:00:03
   --------------- ------------------------ 7.1/18.2 MB 4.6 MB/s eta 0:00:03
   -------------------- ------------------- 9.2/18.2 MB 4.7 MB/s eta 0:00:02
   --------------------- ------------------ 10.0/18.2 MB 4.4 MB/s eta 0:00:02
   ----------------

In [10]:

df["network_status"] = (
    (df["latency_ms"] > 40) |
    (df["packet_loss"] > 1.0) |
    (df["prb_utilization"] > 85) |
    (df["throughput_mbps"] < 60) |          # low throughput = bad experience
    (df["rrc_connected_users"] > 250)       # overload condition
).astype(int)
# Show sample data with new label
df.head()

,cell_id,timestamp,rrc_connected_users,prb_utilization,throughput_mbps,latency_ms,packet_loss,region,technology,band,network_status
0,101,2026-06-13 10:00:00,120,75.5,85.2,18.3,0.2,Riyadh,LTE,1800,0
1,102,2026-06-13 10:00:00,95,60.1,92.5,15.1,0.1,Riyadh,LTE,2100,0
2,103,2026-06-13 10:00:00,180,88.7,70.4,25.6,0.5,Riyadh,NR,3500,1
